In [ ]:
import os
from getpass import getpass
 
# Dictionary لتخزين الـ keys اللي جبناها
# عشان لو طلبنا نفس الـ key تاني منسألش المستخدم
_API_KEYS = {}
 
def get_api_key(key_name: str, display_name: str = None) -> str:
    """
    بتجيب الـ API Key من أي مكان.
   
    Parameters:
    -----------
    key_name : str
        اسم الـ key في النظام (مثلاً 'GOOGLE_API_KEY')
   
    display_name : str, optional
        الاسم اللي يظهر للمستخدم (مثلاً 'Gemini')
   
    Returns:
    --------
    str
        الـ API Key
    """
   
    # لو مفيش display_name، استخدم key_name
    if display_name is None:
        display_name = key_name
   
    # لو الـ key موجود في الـ cache، رجعه
    if key_name in _API_KEYS and _API_KEYS[key_name]:
        return _API_KEYS[key_name]
   
    api_key = None
    source = None
   
    # ────────────────────────────────────────────
    # المحاولة 1: Colab Secrets
    # ────────────────────────────────────────────
    # في Colab، تقدر تحط secrets في الـ sidebar
    # وتجيبها بـ userdata.get()
    try:
        from google.colab import userdata
        api_key = userdata.get(key_name)
        if api_key:
            source = "Colab Secrets"
    except:
        pass  # مش في Colab
   
    # ────────────────────────────────────────────
    # المحاولة 2: Environment Variable
    # ────────────────────────────────────────────
    # لو الـ key محطوط في environment variables
    # زي: export GOOGLE_API_KEY=xxx
    if not api_key:
        api_key = os.environ.get(key_name)
        if api_key:
            source = "Environment Variable"
   
    # ────────────────────────────────────────────
    # المحاولة 3: .env file
    # ────────────────────────────────────────────
    # لو فيه ملف .env في المجلد
    # فيه: GOOGLE_API_KEY=xxx
    if not api_key:
        try:
            from dotenv import load_dotenv
            load_dotenv()  # بتقرأ الـ .env file
            api_key = os.environ.get(key_name)
            if api_key:
                source = ".env file"
        except ImportError:
            pass  # مكتبة python-dotenv مش مثبتة
   
    # ────────────────────────────────────────────
    # المحاولة 4: اسأل المستخدم
    # ────────────────────────────────────────────
    if not api_key:
        print(f"🔑 {display_name} API Key مش موجود.")
        print(f"   جيب الـ key من: https://aistudio.google.com/apikey")
        api_key = getpass(f"   اكتب الـ {display_name} API Key: ")
        source = "Manual Input"
   
    # احفظ الـ key في الـ cache
    _API_KEYS[key_name] = api_key
   
    if source:
        print(f"✅ {display_name} API Key loaded from {source}")
   
    return api_key
 
print("✅ API Key Helper ready!")

In [ ]:
!pip install -q "numpy<2.0.0"
print("✅ numpy < 2.0.0 installed!")
 
!pip install -q langchain-google-genai  langchain-community langchain-text-splitters chromadb pypdf python-dotenv
print("✅ All dependencies installed!")

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

GOOGLE_API_KEY = get_api_key("GOOGLE_API_KEY", "Gemini")
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)
print("✅ Google Generative AI Embeddings initialized!")

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

GOOGLE_API_KEY = get_api_key('GOOGLE_API_KEY', 'Gemini')

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

print("✅ Embedding model ready!")
print("   Model: gemini-embedding-001")

In [ ]:
text = "BIM is Building information modeling"
vector = embeddings.embed_query(text)
print("✅ Text embedded successfully!")
print(f"vector length = {len(vector)}")
print(f"vector sample = {vector[:5]}")

In [ ]:
import numpy as np

def cosine_similarity(v1, v2):
    """بتحسب cosine similarity بين vector1 و vector2"""
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

In [ ]:
text1 = "BIM is Building information modeling"
text2 = "Building information modeling is BIM"
text3 = "من احد برامج نمذجة معلومات البناء هو الريفيت"

vector1 = embeddings.embed_query(text1)
vector2 = embeddings.embed_query(text2)
vector3 = embeddings.embed_query(text3)
sim_1_2 = cosine_similarity(vector1, vector2)
sim_1_3 = cosine_similarity(vector2, vector3)

print(f"Similarity between text1 and text2: {sim_1_2:.4f} (should be close to 1)")
print(f"Similarity between text1 and text3: {sim_1_3:.4f} (should be close to 0)")

In [ ]:
from langchain_community.vectorstores import Chroma
texts = [
    "BIM stands for Building Information Modeling. It is a digital representation of physical and functional characteristics of a facility.",
    "LOD means Level of Development. LOD 100 is conceptual, LOD 200 is approximate geometry, LOD 300 is precise geometry, LOD 400 is fabrication, LOD 500 is as-built.",
    "Revit is a BIM software developed by Autodesk. It is used for architectural design, structural engineering, and MEP coordination.",
    "IFC stands for Industry Foundation Classes. It is an open file format for BIM data exchange between different software applications.",
    "Clash detection is the process of identifying conflicts between different building systems like structural beams and MEP ducts before construction.",
]

vectorstore = Chroma.from_texts(texts=texts, embedding=embeddings, collection_name="bim_docs_3")

print("✅ Texts added to Chroma vector store!")
print("   Total documents in vector store:", {vectorstore._collection.count()})

In [ ]:
from langchain_community.vectorstores import Chroma
texts = [
    "BIM stands for Building Information Modeling. It is a digital representation of physical and functional characteristics of a facility.",
    "LOD means Level of Development. LOD 100 is conceptual, LOD 200 is approximate geometry, LOD 300 is precise geometry, LOD 400 is fabrication, LOD 500 is as-built.",
    "Revit is a BIM software developed by Autodesk. It is used for architectural design, structural engineering, and MEP coordination.",
    "IFC stands for Industry Foundation Classes. It is an open file format for BIM data exchange between different software applications.",
    "Clash detection is the process of identifying conflicts between different building systems like structural beams and MEP ducts before construction.",
]

vectorstore = Chroma.from_texts(texts=texts, embedding=embeddings, collection_name="bim_docs_4")

print("✅ Texts added to Chroma vector store!")
print("   Total documents in vector store:", {vectorstore._collection.count()})

In [ ]:
from langchain_core.documents import Document

sample_documents = [
    Document(
        page_content="""Building Information Modeling (BIM) is a process involving the generation
and management of digital representations of physical and functional characteristics of places.
BIMs are files which can be extracted, exchanged or networked to support decision-making regarding
a built asset. BIM software is used by individuals, businesses and government agencies who plan,
design, construct, operate and maintain buildings and diverse physical infrastructures.""",
        metadata={"page": 1, "source": "BIM_Guide.pdf"}
    ),
    Document(
        page_content="""Level of Development (LOD) is a framework that enables AEC practitioners to specify
and articulate with a high level of clarity the content and reliability of Building Information Models.
LOD 100 represents a conceptual design. LOD 200 shows approximate geometry. LOD 300 provides precise
geometry suitable for construction documents. LOD 400 includes fabrication details. LOD 500 represents
the as-built condition of the facility.""",
        metadata={"page": 2, "source": "BIM_Guide.pdf"}
    ),
    Document(
        page_content="""Clash detection is a critical process in BIM coordination. It identifies conflicts
between different building systems before construction begins. For example, a structural beam might
conflict with an HVAC duct, or electrical conduits might pass through a structural column. Software
like Navisworks and Solibri can automatically detect these clashes, saving significant time and cost
during construction.""",
        metadata={"page": 3, "source": "BIM_Guide.pdf"}
    ),
]

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = text_splitter.split_documents(sample_documents)

print("✅ Sample documents split into chunks!")
print(f"original documents: {len(sample_documents)}")
print(f"split chunks: {len(chunks)}")
for i, chunk in enumerate(chunks[:3], 1):
    print(f"   Chunk {i} metadata: {chunk.metadata}")
    print(f"   Chunk {i} content (truncated): {chunk.page_content[:100]} ...")

In [ ]:
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, collection_name="bim_docs_5")

print("✅ Chroma vector store created from document chunks!")
print("   Total documents in vector store:", {vectorstore._collection.count()})

In [ ]:
q = "what is LOD in BIM?"

result = vectorstore.similarity_search(q, k=3)
print("✅ Similarity search completed!")
print("   Top 3 similar documents:")
for i, doc in enumerate(result, 1):
    print(f"   {i}. {doc.page_content}")

In [ ]:
questions = [
    "what is LOD in BIM?",
    "what is the difference between LOD 300 and LOD 400?",
    "what is the best software for BIM modeling?"]
for q in questions:
    result = vectorstore.similarity_search(q, k=1)
    print(f"✅ Similarity search for question: '{q}'")
    print("   Top 1 similar documents:")
    for i, doc in enumerate(result, 1):
        print(f"   {i}. {doc.page_content}")